# Step 3-A: 선형 함수 근사 (Linear Function Approximation) 가이드

## 1. Q-Table의 한계와 함수 근사 (Function Approximation)
Phase 2에서 경험했듯이, 보드가 커지면 가능한 상태의 수가 수천억 개로 폭발하여 표(Table)에 일일이 적어둘 수 없습니다.
**함수 근사**는 모든 상태를 암기하는 대신, 보드를 요약하는 **몇 가지 핵심 특징(Feature)**을 뽑아내어 가치를 추론하는 기법입니다.

$$\phi(s, a) = \begin{bmatrix} f_0(s, a) \\ f_1(s, a) \\ f_2(s, a) \\ f_3(s, a) \\ f_4(s, a) \end{bmatrix}, \quad Q(s, a) = \mathbf{w}^T \phi(s, a) = \sum_{i} w_i f_i$$

## 2. 우리가 설계한 5가지 핵심 특징 벡터 $\phi(s, a)$
1. **$f_0$ (편향, Bias)**: 기본 상수 `1.0`
2. **$f_1$ (거리 격차)**: `opp_dist - my_dist` (상대가 가야 할 거리 - 내가 가야 할 거리). 내가 유리할수록 큰 양수.
3. **$f_2$ (내 남은 거리 음수화)**: `-my_dist`. 목적지에 도달할수록(거리가 0이 될수록) 값이 커짐.
4. **$f_3$ (상대 남은 거리)**: `opp_dist`. 상대가 목적지에서 멀수록 큰 양수.
5. **$f_4$ (남은 벽 수 차이)**: `my_walls - opp_walls`. 자원(벽) 우위.

```python
# features 벡터 생성 예시:
features[0] = 1.0
features[1] = opp_dist - my_dist
features[2] = -my_dist
features[3] = opp_dist
features[4] = float(temp_env.state.walls_remaining[p] - temp_env.state.walls_remaining[opp])
```

## 3. Semi-Gradient TD 업데이트 수식
선형 Q-러닝에서는 오차(Error)를 계산하여 가중치 벡터 $\mathbf{w}$를 갱신합니다.
- 목표값(Target):
  - 게임 종료(`done == True`): `target = reward`
  - 게임 진행 중(`done == False`): `target = reward - gamma * next_max_q` (제로섬 게임이므로 부호 반전)
- 오차: $\delta = \text{target} - Q(s, a) = \text{target} - \mathbf{w}^T \phi$
- 가중치 갱신: $\mathbf{w} \leftarrow \mathbf{w} + \alpha \cdot \delta \cdot \phi$

```python
current_q = self.get_q_value(phi)
target = reward if done else reward - self.gamma * next_max_q
error = target - current_q
self.weights += self.alpha * error * phi
```

## 4. 왜 Linear Q가 훌륭한가? (가중치 해석)
학습이 끝나면 AI가 스스로 깨우친 가중치 $\mathbf{w}$를 직접 출력하여 눈으로 볼 수 있습니다!
예를 들어 $w_1$ (거리 격차 가중치)이 높은 양수로 수렴했다면, AI가 **"쿼리도에서는 상대와 나의 거리 격차를 벌리는 것이 가장 중요한 승리 비결이다!"**라는 법칙을 학습을 통해 스스로 터득했다는 명확한 증거가 됩니다.